In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             matthews_corrcoef, roc_auc_score)

In [2]:
# load data and remove duplicates
df = pd.read_csv("yeast.csv")
df = df.drop_duplicates()

In [3]:
x = df.drop(columns=["name"])  # input features
y = df["name"]                 # target
 
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
def run(max_depth=None, min_samples_leaf=1, class_weight=None):
    acc_list, f1_list, recall_list, mcc_list, auc_list = [], [], [], [], []
 
    for train_index, test_index in kf.split(x, y):
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
 
        model = DecisionTreeClassifier(max_depth=max_depth,
                                       min_samples_leaf=min_samples_leaf,
                                       class_weight=class_weight,
                                       random_state=42)
        model.fit(x_train, y_train)
 
        y_pred = model.predict(x_test)
        y_prob = model.predict_proba(x_test)
 
        acc_list.append(accuracy_score(y_test, y_pred))
        f1_list.append(f1_score(y_test, y_pred, average="macro"))
        recall_list.append(recall_score(y_test, y_pred, average="macro"))
        mcc_list.append(matthews_corrcoef(y_test, y_pred))
        auc_list.append(roc_auc_score(y_test, y_prob, multi_class="ovr",
                                      average="macro", labels=model.classes_))
 
    print("max_depth =", max_depth, "| min_samples_leaf =", min_samples_leaf,
          "| class_weight =", class_weight)
    print("  Accuracy:", round(sum(acc_list) / 5, 4))
    print("  F1 (macro):", round(sum(f1_list) / 5, 4))
    print("  Recall (macro):", round(sum(recall_list) / 5, 4))
    print("  MCC:", round(sum(mcc_list) / 5, 4))
    print("  AUC (ovr):", round(sum(auc_list) / 5, 4))
    print()

# original tree 
run()

# pruned trees 
run(max_depth=5)
run(max_depth=6, min_samples_leaf=5)
run(max_depth=8, min_samples_leaf=5)
run(max_depth=6, min_samples_leaf=5, class_weight="balanced")

max_depth = None | min_samples_leaf = 1 | class_weight = None
  Accuracy: 0.4976
  F1 (macro): 0.3752
  Recall (macro): 0.3835
  MCC: 0.3559
  AUC (ovr): 0.659

max_depth = 5 | min_samples_leaf = 1 | class_weight = None
  Accuracy: 0.5644
  F1 (macro): 0.4059
  Recall (macro): 0.4089
  MCC: 0.4391
  AUC (ovr): 0.72

max_depth = 6 | min_samples_leaf = 5 | class_weight = None
  Accuracy: 0.5671
  F1 (macro): 0.4103
  Recall (macro): 0.4126
  MCC: 0.4403
  AUC (ovr): 0.7222

max_depth = 8 | min_samples_leaf = 5 | class_weight = None
  Accuracy: 0.5464
  F1 (macro): 0.4052
  Recall (macro): 0.4106
  MCC: 0.4174
  AUC (ovr): 0.7386

max_depth = 6 | min_samples_leaf = 5 | class_weight = balanced
  Accuracy: 0.554
  F1 (macro): 0.514
  Recall (macro): 0.5462
  MCC: 0.4363
  AUC (ovr): 0.794

